# Solutions: Spatial Diagnostic Tests

**Notebook**: 04_spatial_tests.ipynb  
**Date**: 2026-02-22

This notebook provides complete solutions for Exercises 1-3 from the Spatial Diagnostic Tests tutorial.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Local utilities
import sys

from scipy import stats

# PanelBox imports
import panelbox
from panelbox.diagnostics.spatial_tests import (
    LocalMoranI,
    MoranIPanelTest,
    run_lm_tests,
)
from panelbox.models.static import PooledOLS

BASE_DIR = Path("..")
sys.path.insert(0, str(BASE_DIR / "utils"))
from spatial_helpers import (
    build_weight_matrix,
    lm_decision_tree_summary,
    plot_lisa_map,
    plot_moran_scatterplot,
)

# Configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
np.random.seed(42)

# Paths
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "spatial"
TABLES_DIR = OUTPUT_DIR / "tables" / "spatial"
RESULTS_DIR = OUTPUT_DIR / "results" / "spatial"

for d in [FIGURES_DIR, TABLES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"PanelBox version: {panelbox.__version__}")

In [ ]:
# Load all datasets needed for solutions
counties = pd.read_csv(DATA_DIR / "spatial" / "us_counties.csv")
coords_us = pd.read_csv(DATA_DIR / "spatial" / "coordinates_us.csv")
W_counties = np.load(DATA_DIR / "spatial" / "W_counties.npy")

eu_regions = pd.read_csv(DATA_DIR / "spatial" / "eu_regions.csv")
coords_eu = pd.read_csv(DATA_DIR / "spatial" / "coordinates_eu.csv")
W_eu = np.load(DATA_DIR / "spatial" / "W_eu_contiguity.npy")

# Sort panel data
df_panel = counties.sort_values(["county_id", "year"]).reset_index(drop=True)
eu_sorted = eu_regions.sort_values(["region_id", "year"]).reset_index(drop=True)

print(f"US Counties: {counties.shape}")
print(f"EU Regions: {eu_regions.shape}")

---

## Exercise 1 Solution: Build Weight Matrix and Compute Moran's I

**Task**: Construct a spatial weight matrix and test for spatial autocorrelation in income.

In [ ]:
# Step 1: Load coordinates and build KNN weight matrix (k=5)
coords = pd.read_csv(DATA_DIR / "spatial" / "coordinates_us.csv")
coords_array = coords[["latitude", "longitude"]].values

W_knn = build_weight_matrix(coords_array, method="knn", k=5)

print("Weight Matrix Summary")
print("=" * 60)
print(f"Dimensions: {W_knn.shape}")
print(f"Row-normalized: {np.allclose(W_knn.sum(axis=1), 1.0)}")
print(f"Diagonal is zero: {W_knn.diagonal().sum() == 0}")

n_neighbors = (W_knn > 0).sum(axis=1)
total_nonzero = (W_knn > 0).sum()
sparsity = 1 - total_nonzero / W_knn.size

print(f"Average neighbors per unit: {n_neighbors.mean():.1f}")
print(f"Min neighbors: {n_neighbors.min()}")
print(f"Max neighbors: {n_neighbors.max()}")
print(f"Non-zero entries: {total_nonzero} / {W_knn.size}")
print(f"Sparsity: {sparsity:.2%}")

In [ ]:
# Step 2: Moran's I for log_income in 2019
df_2019 = df_panel[df_panel["year"] == 2019].copy()
income_values = df_2019["log_income"].values

# Using MoranIPanelTest on single cross-section
moran_income = MoranIPanelTest(
    residuals=income_values,
    W=W_knn,
    entity_ids=df_2019["county_id"].values,
    time_ids=df_2019["year"].values,
)
result_income = moran_income.run(method="pooled")

print("Moran's I for Log Income (2019)")
print("=" * 60)
print(f"Moran's I: {result_income.statistic:.4f}")
print(f"Expected I: {result_income.expected_value:.4f}")
print(f"Variance: {result_income.variance:.6f}")
print(f"Z-score: {result_income.z_score:.4f}")
print(f"P-value: {result_income.pvalue:.4f}")
print(f"Conclusion: {result_income.conclusion}")

In [ ]:
# Step 3: Moran scatterplot
fig = plot_moran_scatterplot(
    values=income_values,
    W=W_knn,
    entity_ids=df_2019["county_id"].values,
    variable_name="Log Income",
    annotate_outliers=True,
    save_path=str(FIGURES_DIR / "04_ex1_moran_scatter_income.png"),
)
plt.show()

In [ ]:
# Step 4: Interpretation
print("Exercise 1 -- Interpretation")
print("=" * 60)

if result_income.pvalue < 0.05:
    direction = "positive" if result_income.statistic > 0 else "negative"
    print(f"Result: SIGNIFICANT {direction} spatial autocorrelation in log income.")
    print(
        f"  I = {result_income.statistic:.4f} (z = {result_income.z_score:.2f}, p = {result_income.pvalue:.4f})"
    )
    print()
    if result_income.statistic > 0:
        print("Interpretation: Counties with high income tend to be surrounded by")
        print("other high-income counties. This is consistent with economic")
        print("agglomeration effects, where prosperous regions create spillovers")
        print("that benefit their neighbors.")
    else:
        print("Interpretation: Negative autocorrelation suggests a checkerboard pattern.")
else:
    print("Result: No significant spatial autocorrelation at 5% level.")
    print(
        f"  I = {result_income.statistic:.4f} (z = {result_income.z_score:.2f}, p = {result_income.pvalue:.4f})"
    )
    print()
    print("Interpretation: Income does not show strong spatial clustering")
    print("using the KNN(k=5) weight matrix. This could mean:")
    print("  a) Income variation is driven by local factors rather than geography")
    print("  b) The weight matrix choice may not capture the relevant spatial structure")
    print("  c) The signal is present but weak (try a different k or W construction)")

print()
print("The Moran scatterplot shows the distribution across quadrants:")
z = (income_values - income_values.mean()) / income_values.std()
Wz = W_knn @ z
n_hh = ((z >= 0) & (Wz >= 0)).sum()
n_ll = ((z < 0) & (Wz < 0)).sum()
n_hl = ((z >= 0) & (Wz < 0)).sum()
n_lh = ((z < 0) & (Wz >= 0)).sum()
N = len(z)
print(f"  HH: {n_hh} ({100 * n_hh / N:.1f}%)")
print(f"  LL: {n_ll} ({100 * n_ll / N:.1f}%)")
print(f"  HL: {n_hl} ({100 * n_hl / N:.1f}%)")
print(f"  LH: {n_lh} ({100 * n_lh / N:.1f}%)")

---

## Exercise 2 Solution: LM Test Decision Tree

**Task**: Apply full LM test battery and choose the appropriate spatial model.

In [ ]:
# Step 1: OLS model with log_income as dependent variable
model_ex2 = PooledOLS(
    "log_income ~ education_pct + manufacturing_share",
    df_panel,
    entity_col="county_id",
    time_col="year",
)
result_ex2 = model_ex2.fit()

print("OLS Baseline: log_income ~ education_pct + manufacturing_share")
print("=" * 60)
print(result_ex2.summary())

In [ ]:
# Step 2: Run all four LM tests with queen contiguity W
print("LM Tests with Queen Contiguity W")
print("=" * 60)

lm_cont = run_lm_tests(result_ex2, W_counties, alpha=0.05)
display(lm_cont["summary"])
print(f"\nRecommendation: {lm_cont['recommendation']}")
print(f"Reason: {lm_cont['reason']}")

In [ ]:
# Run LM tests with KNN (k=5) W for comparison
print("LM Tests with KNN (k=5) W")
print("=" * 60)

W_knn5_ex2 = build_weight_matrix(coords_array, method="knn", k=5)
lm_knn = run_lm_tests(result_ex2, W_knn5_ex2, alpha=0.05)
display(lm_knn["summary"])
print(f"\nRecommendation: {lm_knn['recommendation']}")
print(f"Reason: {lm_knn['reason']}")

In [ ]:
# Step 3: Decision tree walkthrough for both W specifications
print("Decision Tree -- Queen Contiguity")
print(lm_decision_tree_summary(lm_cont, alpha=0.05))

print("\n\n")
print("Decision Tree -- KNN (k=5)")
print(lm_decision_tree_summary(lm_knn, alpha=0.05))

In [ ]:
# Step 4: Estimate spatial model if recommended
recommendation = lm_cont["recommendation"]
print(f"Primary recommendation (contiguity): {recommendation}")
print("=" * 60)

# Try SAR model
try:
    from panelbox.models.spatial import SpatialLag

    sar_ex2 = SpatialLag(
        "log_income ~ education_pct + manufacturing_share",
        df_panel,
        entity_col="county_id",
        time_col="year",
        W=W_counties,
    )
    sar_result_ex2 = sar_ex2.fit(effects="pooled", method="qml")
    print("\nSAR Model Results:")
    print(sar_result_ex2.summary())

    # Compare OLS vs SAR coefficients
    print("\n\nCoefficient Comparison: OLS vs SAR")
    print("=" * 60)
    comp_df = pd.DataFrame(
        {
            "OLS": result_ex2.params,
            "OLS SE": result_ex2.std_errors,
            "SAR": sar_result_ex2.params,
            "SAR SE": sar_result_ex2.bse
            if hasattr(sar_result_ex2, "bse")
            else sar_result_ex2.std_errors,
        }
    )
    display(comp_df)

except Exception as e:
    print(f"Could not estimate SAR model: {e}")
    print("\nFalling back to OLS comparison only.")

# Try SEM model
try:
    from panelbox.models.spatial import SpatialError

    sem_ex2 = SpatialError(
        "log_income ~ education_pct + manufacturing_share",
        df_panel,
        entity_col="county_id",
        time_col="year",
        W=W_counties,
    )
    sem_result_ex2 = sem_ex2.fit(effects="pooled", method="gmm")
    print("\n\nSEM Model Results:")
    print(sem_result_ex2.summary())

except Exception as e:
    print(f"Could not estimate SEM model: {e}")

In [ ]:
# Discussion
print("Exercise 2 -- Discussion")
print("=" * 60)
print(f"\nContiguity W recommendation: {lm_cont['recommendation']}")
print(f"KNN (k=5) W recommendation: {lm_knn['recommendation']}")
print()

if lm_cont["recommendation"] == lm_knn["recommendation"]:
    print("Both W specifications lead to the SAME recommendation.")
    print("This increases confidence in the spatial model choice.")
else:
    print("The two W specifications lead to DIFFERENT recommendations.")
    print("This suggests sensitivity to the spatial structure definition.")
    print("Consider: the contiguity W captures administrative proximity,")
    print("while KNN captures geographic proximity. Economic theory should")
    print("guide the choice of which is more appropriate.")

print()
print("How much does spatial correlation matter?")
print(f"  OLS R-squared: {result_ex2.rsquared:.4f}")
print(f"  LM-Lag statistic: {lm_cont['lm_lag'].statistic:.4f} (p={lm_cont['lm_lag'].pvalue:.4f})")
print(
    f"  LM-Error statistic: {lm_cont['lm_error'].statistic:.4f} (p={lm_cont['lm_error'].pvalue:.4f})"
)
print()
print("Even if spatial effects are small in magnitude, ignoring them")
print("biases standard errors and can lead to incorrect inference.")

# Save comparison table
comparison = pd.DataFrame(
    [
        {
            "W": "Contiguity",
            "LM-Lag": lm_cont["lm_lag"].statistic,
            "LM-Lag p": lm_cont["lm_lag"].pvalue,
            "LM-Error": lm_cont["lm_error"].statistic,
            "LM-Error p": lm_cont["lm_error"].pvalue,
            "Recommendation": lm_cont["recommendation"],
        },
        {
            "W": "KNN(k=5)",
            "LM-Lag": lm_knn["lm_lag"].statistic,
            "LM-Lag p": lm_knn["lm_lag"].pvalue,
            "LM-Error": lm_knn["lm_error"].statistic,
            "LM-Error p": lm_knn["lm_error"].pvalue,
            "Recommendation": lm_knn["recommendation"],
        },
    ]
)
comparison.to_csv(TABLES_DIR / "04_ex2_lm_comparison.csv", index=False)
display(comparison)

---

## Exercise 3 Solution: LISA Cluster Analysis

**Task**: Identify spatial clusters and outliers in EU R&D expenditure.

In [ ]:
# Step 1: LISA for rd_expenditure (latest year)
latest_year = eu_regions["year"].max()
eu_latest = eu_sorted[eu_sorted["year"] == latest_year].copy()

print(f"LISA Analysis: R&D Expenditure ({latest_year})")
print(f"N regions: {len(eu_latest)}")
print(
    f"R&D expenditure range: [{eu_latest['rd_expenditure'].min():.4f}, {eu_latest['rd_expenditure'].max():.4f}]"
)
print(f"Mean: {eu_latest['rd_expenditure'].mean():.4f}")
print()

print("Computing LISA (499 permutations)...")
lisa_rd = LocalMoranI(
    values=eu_latest["rd_expenditure"].values,
    W=W_eu,
    entity_ids=eu_latest["region_id"].values,
)
lisa_rd_result = lisa_rd.run(permutations=499)

print(lisa_rd_result.summary())

In [ ]:
# Step 2: Identify all four cluster types
rd_clusters = lisa_rd_result.get_clusters(alpha=0.05)

# Merge with region data
rd_clusters = rd_clusters.merge(
    eu_latest[["region_id", "country", "rd_expenditure", "gdp_per_capita", "infrastructure"]],
    left_on="entity_id",
    right_on="region_id",
    how="left",
)

print("Cluster Distribution")
print("=" * 60)
cluster_counts = rd_clusters["cluster_type"].value_counts()
for ctype, count in cluster_counts.items():
    pct = 100 * count / len(rd_clusters)
    print(f"  {ctype}: {count} ({pct:.1f}%)")

print("\nSignificant Clusters Detail:")
sig = rd_clusters[rd_clusters["cluster_type"] != "Not significant"]
if len(sig) > 0:
    for ctype in ["HH", "LL", "HL", "LH"]:
        subset = sig[sig["cluster_type"] == ctype]
        if len(subset) > 0:
            print(f"\n  {ctype} ({len(subset)} regions):")
            print(f"    Countries: {subset['country'].unique().tolist()}")
            print(f"    Mean R&D: {subset['rd_expenditure'].mean():.4f}")
            print(f"    Regions: {subset['entity_id'].tolist()}")
else:
    print("  No significant clusters at alpha = 0.05")
    print("  Note: This can happen when spatial autocorrelation is weak")

In [ ]:
# Step 3: LISA cluster map
eu_coords_array = coords_eu[["latitude", "longitude"]].values

fig = plot_lisa_map(
    local_i=lisa_rd_result.local_i,
    pvalues=lisa_rd_result.pvalues,
    z_values=lisa_rd_result.z_values,
    Wz_values=lisa_rd_result.Wz_values,
    coordinates=eu_coords_array,
    alpha=0.05,
    save_path=str(FIGURES_DIR / "04_ex3_lisa_rd_expenditure.png"),
)
plt.suptitle(
    f"LISA Cluster Map: R&D Expenditure ({latest_year})", fontsize=14, fontweight="bold", y=1.02
)
plt.show()

# Also make Moran scatterplot for context
fig = plot_moran_scatterplot(
    values=eu_latest["rd_expenditure"].values,
    W=W_eu,
    entity_ids=eu_latest["region_id"].values,
    variable_name="R&D Expenditure",
    annotate_outliers=True,
    save_path=str(FIGURES_DIR / "04_ex3_moran_scatter_rd.png"),
)
plt.show()

In [ ]:
# Step 4: Compare characteristics of HH vs LL clusters
print("Comparison of HH vs LL Cluster Characteristics")
print("=" * 60)

hh_rd = rd_clusters[rd_clusters["cluster_type"] == "HH"]
ll_rd = rd_clusters[rd_clusters["cluster_type"] == "LL"]
ns_rd = rd_clusters[rd_clusters["cluster_type"] == "Not significant"]

comparison_rows = []
for label, subset in [
    ("HH (R&D clusters)", hh_rd),
    ("LL (R&D deserts)", ll_rd),
    ("Not significant", ns_rd),
]:
    if len(subset) > 0:
        comparison_rows.append(
            {
                "Group": label,
                "N regions": len(subset),
                "Mean R&D": subset["rd_expenditure"].mean(),
                "Mean GDP/cap": subset["gdp_per_capita"].mean(),
                "Mean Infrastructure": subset["infrastructure"].mean(),
            }
        )

comp_table = pd.DataFrame(comparison_rows)
display(comp_table)
comp_table.to_csv(TABLES_DIR / "04_ex3_cluster_comparison.csv", index=False)

In [ ]:
# Step 5: Economic interpretation
print("Exercise 3 -- Economic Interpretation")
print("=" * 60)
print()

n_hh = len(hh_rd)
n_ll = len(ll_rd)

print(f"Summary of R&D Spatial Clusters in EU Regions ({latest_year}):")
print()

if n_hh > 0:
    print(f"1. R&D HOT SPOTS (HH clusters): {n_hh} regions")
    print("   These are regions with high R&D expenditure surrounded by")
    print("   other high-R&D regions. They represent innovation hubs where")
    print("   knowledge spillovers reinforce R&D activity.")
    print(f"   Countries: {hh_rd['country'].unique().tolist() if n_hh > 0 else 'N/A'}")
    print()

if n_ll > 0:
    print(f"2. R&D DESERTS (LL clusters): {n_ll} regions")
    print("   These regions have low R&D spending and are surrounded by")
    print("   other low-R&D regions. They lack the critical mass of")
    print("   innovative activity to generate spillovers.")
    print(f"   Countries: {ll_rd['country'].unique().tolist() if n_ll > 0 else 'N/A'}")
    print()

print("3. POLICY IMPLICATIONS:")
print("   a) R&D clusters benefit from agglomeration -- policies should")
print("      reinforce existing clusters rather than dispersing resources.")
print("   b) LL regions need targeted support to build initial R&D capacity.")
print("   c) Cross-border cooperation between HH and neighboring regions")
print("      can help diffuse innovation benefits.")
print("   d) Weight matrix choice matters: contiguity-based W captures")
print("      geographic spillovers; an economic-distance W might reveal")
print("      different clusters based on industrial similarity.")

---

## Bonus: Comprehensive Visualization

Let's create a combined figure summarizing all key spatial diagnostics.

In [ ]:
# Comprehensive 2x2 figure
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Panel 1: County unemployment map
df_2019_coords = df_panel[df_panel["year"] == 2019].merge(coords_us, on="county_id")
sc1 = axes[0, 0].scatter(
    df_2019_coords["longitude"],
    df_2019_coords["latitude"],
    c=df_2019_coords["unemployment"],
    cmap="YlOrRd",
    s=20,
    alpha=0.8,
)
axes[0, 0].set_title("US County Unemployment (2019)", fontweight="bold")
axes[0, 0].set_xlabel("Longitude")
axes[0, 0].set_ylabel("Latitude")
plt.colorbar(sc1, ax=axes[0, 0], label="Unemployment Rate")

# Panel 2: Moran scatterplot
unemp = df_2019_coords["unemployment"].values
z_u = (unemp - unemp.mean()) / unemp.std()
Wz_u = W_counties @ z_u
slope = float(z_u @ Wz_u / (z_u @ z_u))

quad_colors = []
for i in range(len(z_u)):
    if z_u[i] >= 0 and Wz_u[i] >= 0:
        quad_colors.append("#d32f2f")
    elif z_u[i] < 0 and Wz_u[i] < 0:
        quad_colors.append("#1976d2")
    elif z_u[i] >= 0 and Wz_u[i] < 0:
        quad_colors.append("#ff9800")
    else:
        quad_colors.append("#4fc3f7")

axes[0, 1].scatter(z_u, Wz_u, c=quad_colors, alpha=0.5, s=20)
x_line = np.linspace(z_u.min() - 0.5, z_u.max() + 0.5, 50)
axes[0, 1].plot(x_line, slope * x_line, "k--", linewidth=1.5, label=f"Slope={slope:.3f}")
axes[0, 1].axhline(0, color="grey", linewidth=0.5)
axes[0, 1].axvline(0, color="grey", linewidth=0.5)
axes[0, 1].set_title("Moran Scatterplot", fontweight="bold")
axes[0, 1].set_xlabel("Standardized Unemployment")
axes[0, 1].set_ylabel("Spatial Lag of Unemployment")
axes[0, 1].legend()

# Panel 3: LM test bar chart
test_names = ["LM-Lag", "LM-Error", "Robust\nLM-Lag", "Robust\nLM-Error"]
test_stats = [
    lm_cont["lm_lag"].statistic,
    lm_cont["lm_error"].statistic,
    lm_cont["robust_lm_lag"].statistic,
    lm_cont["robust_lm_error"].statistic,
]
test_pvals = [
    lm_cont["lm_lag"].pvalue,
    lm_cont["lm_error"].pvalue,
    lm_cont["robust_lm_lag"].pvalue,
    lm_cont["robust_lm_error"].pvalue,
]
bar_colors = ["forestgreen" if p < 0.05 else "lightgrey" for p in test_pvals]

axes[1, 0].bar(test_names, test_stats, color=bar_colors, edgecolor="white")
axes[1, 0].set_title("LM Test Statistics (log_income model)", fontweight="bold")
axes[1, 0].set_ylabel("Test Statistic")
axes[1, 0].axhline(
    stats.chi2.ppf(0.95, 1),
    color="red",
    linestyle="--",
    alpha=0.5,
    label=f"chi2(1) critical = {stats.chi2.ppf(0.95, 1):.2f}",
)
axes[1, 0].legend(fontsize=9)

# Panel 4: Moran's I over time
model_ts = PooledOLS(
    "unemployment ~ log_income + log_population + manufacturing_share",
    df_panel,
    "county_id",
    "year",
)
result_ts = model_ts.fit()
moran_ts = MoranIPanelTest(
    result_ts.resid, W_counties, df_panel["county_id"].values, df_panel["year"].values
)
bp = moran_ts.run(method="by_period")
yrs = sorted(bp.keys())
I_vals = [bp[y].statistic for y in yrs]

axes[1, 1].plot([int(y) for y in yrs], I_vals, "o-", linewidth=2, color="steelblue")
axes[1, 1].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[1, 1].fill_between([int(y) for y in yrs], 0, I_vals, alpha=0.15)
axes[1, 1].set_title("Moran's I Over Time (residuals)", fontweight="bold")
axes[1, 1].set_xlabel("Year")
axes[1, 1].set_ylabel("Moran's I")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_solutions_summary.png", dpi=300, bbox_inches="tight")
plt.show()

print("Summary figure saved.")

---

## Summary of Key Findings

**Exercise 1**: Moran's I reveals the spatial structure of income across US counties. The Moran scatterplot provides a visual decomposition into spatial clusters (HH, LL) and outliers (HL, LH).

**Exercise 2**: The LM decision tree systematically guides model selection. Results can be sensitive to the choice of weight matrix -- always check robustness.

**Exercise 3**: LISA identifies specific regions that drive spatial patterns. R&D clusters (HH) reflect innovation hubs with knowledge spillovers, while R&D deserts (LL) need targeted policy intervention.